In [1]:
from brian2 import *
import os
import sys
import numpy as np
from IPython.display import clear_output, display
import shutil

sys.path = [p for p in sys.path if 'Neuron and Synapse Models' not in p and 'Tools' not in p]
os.chdir(os.path.dirname(os.getcwd()))  # Change to the parent directory
sys.path.append('Neuron and Synapse Models')
sys.path.append('Tools')

from neuronModels import *
from ringAttractorClass import *
from plottingTools import *
from utils import compute_firing_rate

import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, interactive_output, FloatSlider, IntSlider, Dropdown, HTML, FloatText, Label, ToggleButton, Button, Output

# Simulation parameters
defaultclock.dt = 0.1*ms

In [2]:
# Code block to allow easy passing of initial values
values_flag = True
if values_flag:
    init_values = {
        'num_neurons': 120,
        'tau_val': 10,
        'sigma_noise_val': 0.1,
        'stimulus_center': 3.14,
        'stimulus_width': 0.5,
        'I0_val': 30,
        'sigma_exc_val': 0.125,
        'sigma_inh_val': 0.25,
        'g_exc_val': 0.875,
        'g_inh_val': -0.475,
        'g_cosine_val': 0.1,
        'w_inh_val':-0.555,
        'duration_val': 2.0,
        'velocity_duration_val': 0.5,
        'connectivity_profile': 'cosine'
    }

In [ ]:
# Create sliders for parameters
num_neurons_slider = IntSlider(
    min=50,
    max=200, 
    step=10, 
    value=init_values['num_neurons'] if values_flag else 120, 
    description='Number of Neurons:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

tau_slider = FloatSlider(
    min=1, 
    max=20, 
    step=1, 
    value=init_values['tau_val'] if values_flag else 10, 
    description='Tau (ms):' , 
    continuous_update=False,
    style={'description_width': '150px'},
)

sigma_noise_slider = FloatSlider(
    min=0.1, 
    max=5, 
    step=0.1, 
    value=init_values['sigma_noise_val'] if values_flag else 1, 
    description='Noise Sigma (mV):' , 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_center_slider = FloatSlider(
    min=0, 
    max=2*pi, 
    step=0.1, 
    value=init_values['stimulus_center'] if values_flag else 0, 
    description='Stimulus Center (rad):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_width_slider = FloatSlider(
    min=0.1, 
    max=2.0, 
    step=0.1, 
    value=init_values['stimulus_width'] if values_flag else 0.5, 
    description='Stimulus Width:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

I0_slider = FloatSlider(
    min=0, 
    max=50, 
    step=5, 
    value=init_values['I0_val'] if values_flag else 30, 
    description='Input Amplitude (mV):' , 
    continuous_update=False,
    style={'description_width': '150px'},
)

sigma_exc_slider = FloatSlider(
    min=0.05,
    max=2.0,
    step=0.01,
    value=init_values['sigma_exc_val'] if values_flag else 0.0875,
    description='Sigma Excitatory:',
    continuous_update=False,
    style={'description_width': '150px'},
)

sigma_inh_slider = FloatSlider(
    min=0.1,
    max=2.0,
    step=0.01,
    value=init_values['sigma_inh_val'] if values_flag else 0.25,
    description='Sigma Inhibitory:',
    continuous_update=False,
    style={'description_width': '150px'},
)

g_exc_slider = FloatSlider(
    min=0.5,
    max=3.0,
    step=0.01,
    value=init_values['g_exc_val'] if values_flag else 1.0,
    description='Gain Excitatory (mV):',
    continuous_update=False,
    style={'description_width': '150px'},
)

g_inh_slider = FloatSlider(
    min=-3.0,
    max=-0.1,
    step=0.01,
    value=init_values['g_inh_val'] if values_flag else -0.475,
    description='Gain Inhibitory (mV):',
    continuous_update=False,
    style={'description_width': '150px'},
)

g_cosine_slider = FloatSlider(
    min=0.0,
    max=10.0,
    step=0.005,
    value=init_values['g_cosine_val'] if values_flag else 1.0,
    description='Gain Cosine (mV):',
    continuous_update=False,
    style={'description_width': '150px'}
)

velocity_input_slider = FloatSlider(
    min=-2.0, 
    max=2.0, 
    step=0.1, 
    value=0.0, 
    description='Velocity Input (mV):' , 
    continuous_update=False,
    style={'description_width': '150px'},
)

w_inh_slider = FloatSlider(
    min=-3.00, 
    max=-0.01, 
    step=0.01, 
    value=init_values['w_inh_val'] if values_flag else -0.52705411, 
    description='Global Inhibition (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

duration_box = FloatText(
    value=init_values['duration_val'] if values_flag else 2,
    description='Simulation Duration (s):',
    style={'description_width': '150px'}
)

input_duration_box = FloatText(
    value=0.5,
    description='Input Duration (s):',
    style={'description_width': '150px'}
)

velocity_duration_box = FloatText(
    value=init_values['velocity_duration_val'] if values_flag else 0.5,
    description='Velocity Duration (s):',
    style={'description_width': '150px'}
)

profile_dropdown = Dropdown(
    options=['mexican_hat', 'gaussian', 'cosine'],
    value=init_values['connectivity_profile'] if values_flag else 'cosine',
    description='Synapse Profile:',
    style={'description_width': '150px'},
)

angles_dropdown = Dropdown(
    options=['degrees', 'radians', 'radians (symbolic)'],
    value='degrees',
    description='Angular Representation:',
    style={'description_width': '150px'},
)

autapse_button = ToggleButton(
    value=False,
    description='Autapse',
    tooltip='Allows autapse connections',
    button_style=''
)

global_inh_button = ToggleButton(
    value=True,
    description='Global Inhibitory Neuron',
    tooltip='Adds a global inhibitory neuron',
    button_style='',
    layout=Layout(width='auto', height='auto')
)

input_type_dropdown = Dropdown(
    options=[
        ('Gaussian Input', 'gaussian'),
        ('Target Neuron Only', 'target'), 
        ('All Neurons', 'all')
    ],
    value='gaussian',
    description='Input Type:',
    style={'description_width': '150px'},
)

target_neuron_slider = IntSlider(
    min=0,
    max=119,
    step=1,
    value=60,
    description='Target Neuron:',
    continuous_update=False,
    style={'description_width': '150px'},
)

# Run button for standalone simulation
run_button = Button(
    description='Run Simulation',
    button_style='success',
    tooltip='Click to run the simulation with C++ standalone mode',
    layout=Layout(width='200px', height='40px')
)

# Output area for visualization
output_area = Output()

target_neuron_checkbox = ToggleButton(
    value=False,
    description='Individual Neuron Input',
    tooltip='Adds input to a single neuron',
    button_style='',
    layout=Layout(width='auto', height='auto')
)

In [ ]:
def run_simulation():
    """
    Run the simulation with current widget parameter values using C++ standalone mode
    Returns a dictionary with simulation results
    """
    # Get parameter values from widgets
    num_neurons = num_neurons_slider.value
    tau_val = tau_slider.value
    sigma_noise_val = sigma_noise_slider.value
    stimulus_center = stimulus_center_slider.value
    stimulus_width = stimulus_width_slider.value
    I0_val = I0_slider.value
    sigma_exc_val = sigma_exc_slider.value
    sigma_inh_val = sigma_inh_slider.value
    g_exc_val = g_exc_slider.value
    g_inh_val = g_inh_slider.value
    g_cosine_val = g_cosine_slider.value
    velocity_input = velocity_input_slider.value
    w_inh_val = w_inh_slider.value
    duration_val = duration_box.value
    input_duration_val = input_duration_box.value
    velocity_duration_val = velocity_duration_box.value
    autapse = autapse_button.value
    global_inh = global_inh_button.value
    syn_profile = profile_dropdown.value
    ticks_angles = angles_dropdown.value
    input_type = input_type_dropdown.value
    target_neuron = target_neuron_slider.value
    target_neuron_checkbox_val = target_neuron_checkbox.value
    
    # Set up build directory for C++ code
    build_dir = 'standalone_build'
    if os.path.exists(build_dir):
        shutil.rmtree(build_dir)
    os.makedirs(build_dir)
    
    # Reset device for new simulation
    device.reinit()
    device.activate()
    
    # Convert slider values to Brian units
    tau = tau_val * ms
    sigma_noise = sigma_noise_val * mV
    V_rest = -70 * mV
    I0 = I0_val * mV
    sim_duration = duration_val * second
    g_exc = g_exc_val * mV
    g_inh = g_inh_val * mV
    g_cosine = g_cosine_val * mV
    w_inh = w_inh_val * mV
    
    # Define neuron positions
    positions = linspace(0, 2*pi, num_neurons, endpoint=False)
    
    # Calculate external input based on input type
    I_ext_array = np.zeros(num_neurons) * mV
    if input_type == 'gaussian':
        # Original Gaussian input
        d = np.angle(np.exp(1j * (positions - stimulus_center)))
        I_ext_array += I0 * np.exp(-(d**2) / (2 * stimulus_width**2))
    elif input_type == 'all':
        # Equal input to all neurons
        I_ext_array += np.ones(num_neurons) * I0
    
    if target_neuron_checkbox_val:
        # Input only to target neuron
        I_ext_array[target_neuron] += I0
    
    # Set up neuron model
    neuron_eq = Equations(LIF_xi_vel_eq, tau=tau, V_rest=V_rest, sigma_noise=sigma_noise)
    
    # Set up ring attractor parameters
    Vth = -48 * mV
    V_reset = -80 * mV
    refractory_period = 5 * ms
    
    # Create the ring attractor network
    ringAttractor = RingAttractor(
        neuron_eq, 
        num_neurons, 
        Vth, V_reset, refractory_period,
        syn_profile=syn_profile,
        autapse=autapse,
        glob_inh=global_inh, w_inh=w_inh,
        g_cosine=g_cosine,
        sigma_exc=sigma_exc_val, sigma_inh=sigma_inh_val,
        g_exc=g_exc, g_inh=g_inh
    )
    
    # Set external input
    ringAttractor.ring_pool.I_ext = I_ext_array
    ringAttractor.ring_pool.I_vel = 0.0 * volt
    
    # Setup monitors
    spikemon = SpikeMonitor(ringAttractor.ring_pool)
    statemon = StateMonitor(ringAttractor.ring_pool, 'V', record=True)
    inputmon = StateMonitor(ringAttractor.ring_pool, 'I_ext', record=True)
    
    # Define network operation for potential clipping
    @network_operation(dt=defaultclock.dt)
    def enforce_lower_bound():
        ringAttractor.ring_pool.V[:] = clip(ringAttractor.ring_pool.V[:], V_reset, inf*volt)
    
    # Set of Brian objects to be added to the network
    localObjects = [enforce_lower_bound, spikemon, statemon, inputmon]
    
    # Add inhibitory neuron monitors if global inhibition is enabled
    if global_inh:
        statemon_inh = StateMonitor(ringAttractor.glob_inh_neuron, 'V', record=True)
        spikemon_inh = SpikeMonitor(ringAttractor.glob_inh_neuron)
        localObjects.extend([statemon_inh, spikemon_inh])
    
    # Create network
    net = Network(ringAttractor.BrianObjects + localObjects)
    
    # Calculate simulation segments
    input_on = input_duration_val * second
    input_off = input_on
    velocity_on = velocity_duration_val * second
    end_duration = sim_duration - input_on - input_off - velocity_on
    
    # Prepare to store values for segment simulation
    stored_values = {'positions': positions, 'I_ext_array': I_ext_array}
    
    # Store functions to execute before each segment
    segment_funcs = []
    
    # First segment: Input on
    segment_funcs.append((lambda: None, input_on))
    
    # Second segment: Input off
    def turn_off_input():
        ringAttractor.ring_pool.I_ext = I_ext_array * 0
    segment_funcs.append((turn_off_input, input_off))
    
    # Third segment: Velocity on
    def turn_on_velocity():
        ringAttractor.ring_synapses_asym.vel_in = velocity_input
    segment_funcs.append((turn_on_velocity, velocity_on))
    
    # Fourth segment: Velocity off
    def turn_off_velocity():
        ringAttractor.ring_synapses_asym.vel_in = 0.0
    segment_funcs.append((turn_off_velocity, end_duration))
    
    # Store state modification times for visualization
    state_changes = [0*second]
    current_time = 0*second
    for _, duration in segment_funcs:
        current_time += duration
        state_changes.append(current_time)
    
    # Prepare network for standalone compilation
    device.build(directory=build_dir, compile=True, run=True, debug=False)
    
    # Run all segments
    for func, duration in segment_funcs:
        func()
        net.run(duration)
    
    # Collect results for visualization
    results = {
        'spikemon': spikemon,
        'statemon': statemon,
        'inputmon': inputmon,
        'positions': positions,
        'I_ext_array': I_ext_array,
        'Vth': Vth,
        'input_on': input_on,
        'sim_duration': sim_duration,
        'num_neurons': num_neurons,
        'state_changes': state_changes
    }
    
    # Add inhibitory neuron results if applicable
    if global_inh:
        results['statemon_inh'] = statemon_inh
        results['spikemon_inh'] = spikemon_inh
    
    return results


def visualize_results(results):
    """Create visualizations from simulation results"""
    # Extract results
    spikemon = results['spikemon']
    statemon = results['statemon']
    positions = results['positions']
    I_ext_array = results['I_ext_array']
    Vth = results['Vth']
    input_on = results['input_on']
    sim_duration = results['sim_duration']
    num_neurons = results['num_neurons']
    
    # Create figure with 6 subplots arranged in 3 rows and 2 columns
    fig = plt.figure(figsize=(15, 15))

    # 1. Input Current Plot
    ax1 = fig.add_subplot(3, 2, 1)
    ax1.plot(positions/(2*pi), I_ext_array/mV)
    ax1.set_title('Input Current')
    ax1.set_xlabel('Position (rad)')
    ax1.set_ylabel('Current (mV)')

    # 2. Raster Plot
    ax2 = fig.add_subplot(3, 2, 2)
    raster_plot(spikemon, ax=ax2, stim_periods=(0*second, input_on),
                stim_display_method='highlight', duration=sim_duration, 
                num_neurons=num_neurons, y_axisFull=True)

    # 3. Firing Rate Profile Plot
    ax3 = fig.add_subplot(3, 2, 3)
    firing_rate, _ = firing_rate_profile(spikemon, positions/(2*pi), input_on, ax=ax3)

    # 4. Polar Plot of the Population Vector Average (PVA)
    ax4 = fig.add_subplot(3, 2, 4, projection='polar')
    polar_plot_PVA(firing_rate, positions, scale=1.2, ax=ax4)

    # 5. Time-Resolved PVA Plot
    ax5 = fig.add_subplot(3, 2, 5)
    _, _ = time_resolved_PVA(spikemon, positions, sim_duration, num_neurons, 
                          window_size=50*ms, ax=ax5, color_windows=True, 
                          stim_periods=(0*second, input_on))

    # 6. Membrane potential traces
    ax6 = fig.add_subplot(3, 2, 6)
    membrane_potential_traces(statemon, sim_duration, Vth=Vth, ax=ax6)
    
    plt.tight_layout()
    plt.show()
    
    # If global inhibition neuron data is available, plot it
    if 'statemon_inh' in results and 'spikemon_inh' in results:
        statemon_inh = results['statemon_inh']
        spikemon_inh = results['spikemon_inh']
        
        fig_inh = plt.figure(figsize=(12, 5))
        
        # Inhibitory neuron membrane potential
        ax1 = fig_inh.add_subplot(1, 2, 1)
        ax1.plot(statemon_inh.t/ms, statemon_inh.V[0]/mV)
        ax1.set_title('Inhibitory Neuron Membrane Potential')
        ax1.set_xlabel('Time (ms)')
        ax1.set_ylabel('Membrane Potential (mV)')
        
        # Inhibitory neuron spikes
        ax2 = fig_inh.add_subplot(1, 2, 2)
        ax2.plot(spikemon_inh.t/ms, spikemon_inh.i, '.k')
        ax2.set_title('Inhibitory Neuron Spikes')
        ax2.set_xlabel('Time (ms)')
        ax2.set_ylabel('Neuron Index')
        
        plt.tight_layout()
        plt.show()

In [5]:
def on_run_button_click(b):
    """Handler for run button clicks"""
    with output_area:
        clear_output(wait=True)
        print("Running simulation with C++ standalone mode...")
        print("This may take a moment...")
        
        try:
            # Run simulation
            results = run_simulation()
            
            # Clear progress message and show results
            clear_output(wait=True)
            visualize_results(results)
            
        except Exception as e:
            clear_output(wait=True)
            print(f"Error in simulation: {str(e)}")
            import traceback
            traceback.print_exc()

# Connect the button click to the handler
run_button.on_click(on_run_button_click)

In [ ]:
# Create the parameter boxes with descriptive titles
box_neurons = VBox([
    HTML(value="<b>Neurons Parameters:</b>"),
    num_neurons_slider,
    tau_slider,
    sigma_noise_slider
], layout=Layout(width='25%', align_items='center'))

box_input = VBox([
    HTML(value="<b>Input Parameters:</b>"),
    input_type_dropdown,
    stimulus_center_slider,
    stimulus_width_slider,
    target_neuron_checkbox,
    target_neuron_slider,
    velocity_input_slider
], layout=Layout(width='25%', align_items='center'))

checkbox_subBox = HBox([
    autapse_button,
    global_inh_button
], layout=Layout(align_items='center'))

box_connectivity = VBox([
    HTML(value="<b>Connectivity Profile Parameters:</b>"),
    checkbox_subBox,
    profile_dropdown
], layout=Layout(width='25%', align_items='center'))

def update_connectivity_box(change):
    if change['new'] == 'cosine':
        box_connectivity.children = [
            HTML(value="<b>Connectivity Profile Parameters:</b>"),
            checkbox_subBox,
            profile_dropdown,
            g_cosine_slider
        ]
    elif change['new'] == 'gaussian':   
        box_connectivity.children = [
            HTML(value="<b>Connectivity Profile Parameters:</b>"),
            checkbox_subBox,
            profile_dropdown
        ]
    else:  # mexican_hat
        box_connectivity.children = [
            HTML(value="<b>Connectivity Profile Parameters:</b>"),
            checkbox_subBox,
            profile_dropdown,
            sigma_exc_slider,
            sigma_inh_slider,
            g_exc_slider,
            g_inh_slider
        ]
    
    if global_inh_button.value:
        box_connectivity.children = list(box_connectivity.children) + [w_inh_slider]

# Add observer for global inhibitory neuron toggle
def update_for_global_inh(change):
    # Call update_connectivity_box to refresh UI with current synapse profile
    update_connectivity_box({'new': profile_dropdown.value})

global_inh_button.observe(update_for_global_inh, names='value')
profile_dropdown.observe(update_connectivity_box, names='value')
update_connectivity_box({'new': profile_dropdown.value})

box_simulation = VBox([
    HTML(value="<b>Simulation Parameters:</b>"),
    input_duration_box,
    velocity_duration_box,
    duration_box,
    angles_dropdown
], layout=Layout(width='25%', align_items='center'))

# Horizontal layout for parameter sections
controls = HBox(
    [box_neurons, box_input, box_connectivity, box_simulation],
    layout=Layout(justify_content='center', margin='20px')
)

# Add run button in its own row
run_button_box = HBox(
    [run_button],
    layout=Layout(justify_content='center', margin='10px')
)

# Complete dashboard with controls, run button, and output area
dashboard = VBox(
    [
        controls, 
        run_button_box,
        output_area
    ],
    layout=Layout(align_items='stretch', justify_content='space-around')
)

display(dashboard)

def update_input_visibility(change):
    if input_type_dropdown.value == 'gaussian':
        stimulus_center_slider.layout.visibility = 'visible'
        stimulus_width_slider.layout.visibility = 'visible'
    else:
        stimulus_center_slider.layout.visibility = 'hidden'
        stimulus_width_slider.layout.visibility = 'hidden'

    if target_neuron_checkbox.value:
        target_neuron_slider.layout.visibility = 'visible'
    else:
        target_neuron_slider.layout.visibility = 'hidden'

input_type_dropdown.observe(update_input_visibility, names='value')
target_neuron_checkbox.observe(update_input_visibility, names='value')

# Initialize visibility state
update_input_visibility({'new': input_type_dropdown.value})